In [1]:
import sys
import numpy as np

from Rain.Rain import Rain

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )

In [3]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        # network parameters
        hidden_units = 256
        dropout = 0.45
        input_size = 784
        num_labels = 10
        # Define the layers
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_units, hidden_units)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_units, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [4]:
model = Model()
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "pytorch",
      "params": {
        "loss": nn.CrossEntropyLoss(),
        "optimizer": optim.Adam(model.parameters(), lr=0.001)

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [5]:
X_train, y_train = get_train_data()
y_train = np.argmax(y_train, axis=1)

In [6]:
rain = Rain(config, model)

2023-07-07 16:41:08,714 [DEBUG] [Rain] Rain is initialized
2023-07-07 16:41:08,716 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 16:41:08,803 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 16:41:08,809 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 16:41:08,817 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 16:41:08,831 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 16:41:08,832 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 16:41:08,833 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [7]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 16:41:08,844 [INFO] [Provisioner] provisioner is serving
2023-07-07 16:41:08,844 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 16:41:08,846 [INFO] [Coordinator] coordinator is serving
2023-07-07 16:41:08,848 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 16:41:08,855 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 16:41:08,857 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-07 16:41:08,858 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-07 16:41:08,859 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 16:41:08,859 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 16:41:08,861 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-07 16:41:08,872 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50

Epoch [1/5], Loss: 0.7276, Accuracy: 0.7841
Epoch [1/5], Loss: 0.7257, Accuracy: 0.7801
Epoch [2/5], Loss: 0.2952, Accuracy: 0.9129
Epoch [2/5], Loss: 0.3076, Accuracy: 0.9074
Epoch [3/5], Loss: 0.2232, Accuracy: 0.9328
Epoch [3/5], Loss: 0.2350, Accuracy: 0.9310
Epoch [4/5], Loss: 0.1816, Accuracy: 0.9457


2023-07-07 16:41:56,377 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 16:41:56,379 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-07 16:41:56,456 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully


Epoch [4/5], Loss: 0.1889, Accuracy: 0.9423


2023-07-07 16:41:57,419 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 16:41:57,423 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
2023-07-07 16:41:57,494 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_1_trained.pkl from worker2 successfully


...

2023-07-07 16:42:05,266 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 16:42:05,268 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_2_trained.pkl from worker1
2023-07-07 16:42:05,270 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 16:42:05,271 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-07 16:42:05,275 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 16:42:05,276 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_2_trained.pkl from worker3
2023-07-07 16:42:05,323 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_2_trained.pkl from worker1 successfully
2023-07-07 16:42:05,326 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from w

Epoch [1/5], Loss: 0.1288, Accuracy: 0.9626
Epoch [2/5], Loss: 0.1059, Accuracy: 0.9673
Epoch [3/5], Loss: 0.0925, Accuracy: 0.9701
Epoch [4/5], Loss: 0.0872, Accuracy: 0.9725


2023-07-07 16:42:10,686 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 16:42:10,687 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
2023-07-07 16:42:10,752 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully


Epoch [5/5], Loss: 0.0828, Accuracy: 0.9746
sending data to divider


2023-07-07 16:42:11,856 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 16:42:11,857 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
2023-07-07 16:42:11,860 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 16:42:11,861 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-07 16:42:11,912 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 16:42:11,912 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-07 16:42:11,926 [DEBUG] [DeepLearning] Iteration 3/3 complete.
2023-07-07 16:42:11,927 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-07 16:42:11,928 [DEBUG] [Divider] Divider stopped serving


In [8]:
def evaluate_model(model, X_test, y_test, batch_size):
    # Convert numpy arrays to PyTorch tensors
    X_test = torch.from_numpy(X_test).float()
    y_test = torch.from_numpy(y_test).long()

    # Create a TensorDataset
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    total_correct = 0
    total_samples = 0
    
    for i, (data, labels) in enumerate(test_loader):
        # Forward pass
        outputs = model(data)

        # Compute training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples

In [9]:
X_test, y_test = get_test_data()
y_test = np.argmax(y_test, axis=1)
acc = evaluate_model(model, X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))


Test accuracy: 96.9%
